In [145]:
# 导入库并初始化客户端（使用 openai-python 接口）
import os
import json
from pathlib import Path
from typing import List, Dict, Any
import time
import random
import pandas as pd
from tqdm import tqdm
import openai

# 获取 API key：优先从环境变量读取。你也可以直接在此处设置 API_KEY（不推荐在共享环境中明文填写）。
# Initialize OpenAI client
API_KEY = "REDACTED_OPENAI_KEY" # 用户提供的 API Key
client = openai.OpenAI(api_key=API_KEY)
# 配置：模型、根目录、输出目录等
MODEL = 'gpt-4o-mini'  # 可替换为你有权限的模型
ROOT_DIR = Path('.')  # 工作区根，包含各个 benchmark 文件夹
MAX_PER_QUESTION = 10
SEED = 42
random.seed(SEED)

print('准备就绪。请确认 API Key 已设置（或在本单元中填写）。')

准备就绪。请确认 API Key 已设置（或在本单元中填写）。


## 构造用于生成参考答案的 Prompt 模板
下面的提示会：
- 给出问题文本和（如存在）已有的参考答案样例；
- 要求生成 `MAX_PER_QUESTION` 条高质量、表达不同但都正确或可被接受的参考答案；
- 要求返回 JSON 数组（便于解析）。

In [146]:
def build_prompt(question_text: str, sample_solution: Any, examples: List[str]=None, n: int = 20) -> str:
    # The examples parameter can include top-level answers from the training set for style guidance (optional)
    header = (f"Generate {n} high-quality, diverse reference answers for the following question. Each answer should be an independent, complete, and concise response that can serve as a scoring reference.\n\n")
    q_block = f"Question:\n{question_text}\n\n"
    sample_block = ''
    if sample_solution:
        # Support list or str
        if isinstance(sample_solution, list):
            sample_block = 'Sample reference answers:\n' + '\n'.join([f'- {s}' for s in sample_solution[:5]]) + '\n\n'
        else:
            sample_block = 'Sample reference answer:\n- ' + str(sample_solution)[:1000] + '\n\n'
    examples_block = ''
    if examples:
        examples_block = 'High-quality student answer examples from the dataset (for style guidance):\n' + '\n'.join([f'- {e[:300]}' for e in examples[:20]]) + '\n\n'
    footer = (
        'Requirements:\n'
        '- The generated answers should correctly address the question (or provide valid alternative perspectives);\n'
        '- Use diverse wording, sentence structures, and focus points to increase variety;\n'
        '- Each answer should be only ONE sentence long, avoiding scoring labels or meta-comments;\n'
        '- Return the answers in JSON format, structured as {"answers": ["...", "...", ...]}, and do not include any additional text.\n'
        '- Ensure the language of the answers matches the language of the question (e.g., answer in German if the question is in German).\n'
    )
    prompt = header + q_block + sample_block + examples_block + footer
    return prompt

# Test prompt construction (does not call the API)
# p = build_prompt('Example question: Briefly describe the basic process of photosynthesis.', 'Photosynthesis is the process by which plants use light energy to convert carbon dioxide and water into glucose and oxygen.', examples=['Plants perform photosynthesis in chloroplasts.'], n=5)
# print(p[:800])

Generate 5 high-quality, diverse reference answers for the following question. Each answer should be an independent, complete, and concise response that can serve as a scoring reference.

Question:
Example question: Briefly describe the basic process of photosynthesis.

Sample reference answer:
- Photosynthesis is the process by which plants use light energy to convert carbon dioxide and water into glucose and oxygen.

High-quality student answer examples from the dataset (for style guidance):
- Plants perform photosynthesis in chloroplasts.

Requirements:
- The generated answers should correctly address the question (or provide valid alternative perspectives);
- Use diverse wording, sentence structures, and focus points to increase variety;
- Each answer should be only ONE sentence long, 


## 调用 LLM 并保存结果的函数（带重试与解析容错）

In [147]:

def call_llm_for_answers(prompt: str, model: str = MODEL, max_tokens: int = 800, temperature: float = 0.8, retries: int = 3) -> List[str]:
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            # ensure client has api key
            if not (API_KEY or os.environ.get("OPENAI_API_KEY")):
                raise RuntimeError("No API key found. Set OPENAI_API_KEY or API_KEY in this notebook.")
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "system", "content": "You are a helpful, precise assistant."},
                          {"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            # robustly extract text
            try:
                text = resp.choices[0].message.content
            except Exception:
                text = resp["choices"][0]["message"]["content"]
            # try JSON parse
            try:
                text = text.strip().lstrip("```json").rstrip("```").strip()
                parsed = json.loads(text)
                if isinstance(parsed, dict) and "answers" in parsed:
                    return parsed["answers"]
                if isinstance(parsed, list):
                    return parsed
            except Exception:
                # fallback: extract lines/numbered items
                lines = [l.strip() for l in text.splitlines() if l.strip()]
                answers = []
                for l in lines:
                    import re
                    m = re.sub(r'^\s*\d+[.)\\s-]*', '', l)
                    if len(m) > 0:
                        answers.append(m)
                if answers:
                    return answers[:MAX_PER_QUESTION]
                return []
            raise ValueError("Unable to parse model output as an answers list")
        except Exception as e:
            last_err = e
            wait = 2 ** attempt
            print(f"LLM call error (attempt {attempt}/{retries}): {e}. Waiting {wait}s before retry")
            time.sleep(wait)
    raise RuntimeError(f"LLM calls failed: {last_err}")

## 批量为某个数据集生成参考答案（示例代码单元）
你可以按需调整 `dataset_folder`（例如 `asap-sas-data`）并运行该单元来为该数据集的所有问题生成参考答案。

In [151]:
def generate_for_dataset(dataset_folder: str, max_per_question: int = MAX_PER_QUESTION, dry_run: bool = False):
    folder = Path(dataset_folder)
    meta_path = folder / 'question_meta.json'
    if not meta_path.exists():
        print(f'未在 {folder} 找到 question_meta.json，跳过')
        return
    with open(meta_path, 'r', encoding='utf-8') as f:
        qmeta = json.load(f)

    results = {}
    question_ids = list(qmeta.keys())
    for qid in tqdm(question_ids, desc=f'Generating {dataset_folder}'):
        info = qmeta.get(str(qid), {})
        question_text = info.get('question', '')
        sample_solution = info.get('sample_solution', info.get('sample_solutions', info.get('reference_answer', None)))

        # 从训练数据和测试数据中抓取少量高分答案作为 style examples（如果存在 train.csv 和 test*.csv）
        examples = []
        csv_files = [folder / 'train.csv'] + list(folder.glob('test*.csv'))
        df_list = []
        for csv_file in csv_files:
            if csv_file.exists():
                try:
                    df_temp = pd.read_csv(csv_file)
                    df_list.append(df_temp)
                except Exception as e:
                    print(f'读取 {csv_file} 失败：{e}')
        if df_list:
            df = pd.concat(df_list, ignore_index=True)
            # 答案列固定为 'answer'
            ans_col = 'answer'
            # 找高分/最后一类 label 的若干答案（如果有 label 列）
    
            labels = df['level'].dropna().astype(int)
            max_label = labels.max()  # Find the highest label
            subset = df[df['level'] == max_label]
            if not subset.empty:
                examples = subset[ans_col].dropna().astype(str).tolist()
       

        prompt = build_prompt(question_text, sample_solution, examples=examples, n=max_per_question)

        if dry_run or not API_KEY:
            # 仅返回 prompt（用于调试）
            print(f'--- DRY RUN: question {qid} prompt preview ---')
            print(prompt)
            results[str(qid)] = {'question_text': question_text, 'sample_solution': sample_solution, 'answers': []}
            continue

        try:
            answers = call_llm_for_answers(prompt, model=MODEL, temperature=0.8)
            # 去重并截取到 max_per_question
            uniq = []
            for a in answers:
                if a not in uniq:
                    uniq.append(a)
            results[str(qid)] = {'question_text': question_text, 'sample_solution': sample_solution, 'answers': uniq[:max_per_question]}
        except Exception as e:
            print(f'Question {qid} 生成失败：{e}')
            results[str(qid)] = {'question_text': question_text, 'sample_solution': sample_solution, 'answers': [], 'error': str(e)}

    out_path = Path(f'{Path(dataset_folder).name}/llm_reference_answers.json')
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f'✓ 已保存：{out_path} （问题数：{len(results)}）')
    return out_path

# 示例（干运行：仅展示 prompt）
generate_for_dataset('beetle', dry_run=False)

Generating beetle: 100%|██████████| 56/56 [03:56<00:00,  4.22s/it]

✓ 已保存：beetle/llm_reference_answers.json （问题数：56）


PosixPath('beetle/llm_reference_answers.json')